<a href="https://colab.research.google.com/github/uwol1116/GAI-Class/blob/main/02_Programming_problem_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#  Problem 2

import torch
import torch.nn as nn
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

transform = transforms.ToTensor()

train_dataset = datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform)
test_dataset  = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

CLASSES = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck')

class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1   = nn.Conv2d(3,  16, kernel_size=3)
        self.relu1   = nn.ReLU()
        self.pool    = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2   = nn.Conv2d(16, 32, kernel_size=3)
        self.relu2   = nn.ReLU()
        self.fc      = nn.Linear(5408, 10)

    def forward(self, x):
        x = self.conv1(x);  print(f"  After Conv1  : {x.shape}")
        x = self.relu1(x)
        x = self.pool(x);   print(f"  After MaxPool: {x.shape}")
        x = self.conv2(x);  print(f"  After Conv2  : {x.shape}")
        x = self.relu2(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

class CNNModelModified(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,  16, kernel_size=3)
        self.relu1 = nn.ReLU()
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3)
        self.relu2 = nn.ReLU()
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3)
        self.relu3 = nn.ReLU()
        self.fc    = nn.Linear(7744, 10)

    def forward(self, x):
        x = self.conv1(x);  print(f"  After Conv1  : {x.shape}")
        x = self.relu1(x)
        x = self.pool(x);   print(f"  After MaxPool: {x.shape}")
        x = self.conv2(x);  print(f"  After Conv2  : {x.shape}")
        x = self.relu2(x)
        x = self.conv3(x);  print(f"  After Conv3  : {x.shape}")
        x = self.relu3(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x


def train(model, loader, epochs=5):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for images, labels in loader:
            optimizer.zero_grad()
            out  = model(images)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"  Epoch {epoch+1}/{epochs}  loss={total_loss/len(loader):.4f}")

def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return correct / total * 100

def show_shapes(model, loader, label):
    print(f"\n[Feature map shapes – {label}]")
    images, _ = next(iter(loader))
    with torch.no_grad():
        _ = model(images[:1])


def predict_three(model, dataset, label):
    print(f"\n[Predictions for 3 test images – {label}]")
    model.eval()
    with torch.no_grad():
        for i in range(3):
            img, true_label = dataset[i]
            out  = model(img.unsqueeze(0))
            pred = out.argmax(dim=1).item()
            print(f"  Image {i+1}: true={CLASSES[true_label]:8s}  pred={CLASSES[pred]}")

EPOCHS = 5

print("=" * 50)
print("Baseline CNNModel")
print("=" * 50)

base_model = CNNModel()
show_shapes(base_model, train_loader, "Baseline (before training)")

print("\n[Training Baseline]")
train(base_model, train_loader, epochs=EPOCHS)

base_acc = evaluate(base_model, test_loader)
print(f"\nBaseline test accuracy: {base_acc:.2f}%")

predict_three(base_model, test_dataset, "Baseline")

print("\n" + "=" * 50)
print("Modified CNNModel  (+ extra Conv3 layer)")
print("=" * 50)

mod_model = CNNModelModified()
show_shapes(mod_model, train_loader, "Modified (before training)")

print("\n[Training Modified]")
train(mod_model, train_loader, epochs=EPOCHS)

mod_acc = evaluate(mod_model, test_loader)
print(f"\nModified test accuracy: {mod_acc:.2f}%")

predict_three(mod_model, test_dataset, "Modified")

print("\n" + "=" * 50)
print("Performance Comparison")
print("=" * 50)
print(f"  Baseline accuracy : {base_acc:.2f}%")
print(f"  Modified accuracy : {mod_acc:.2f}%")
diff = mod_acc - base_acc
direction = "improvement" if diff >= 0 else "decrease"
print(f"  Difference        : {abs(diff):.2f}% {direction}")

100%|██████████| 170M/170M [00:03<00:00, 43.8MB/s]


스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  After Conv1  : torch.Size([64, 16, 30, 30])
  After MaxPool: torch.Size([64, 16, 15, 15])
  After Conv2  : torch.Size([64, 32, 13, 13])
  After Conv3  : torch.Size([64, 64, 11, 11])
  After Conv1  : torch.Size([64, 16, 30, 30])
  After MaxPool: torch.Size([64, 16, 15, 15])
  After Conv2  : torch.Size([64, 32, 13, 13])
  After Conv3  : torch.Size([64, 64, 11, 11])
  After Conv1  : torch.Size([64, 16, 30, 30])
  After MaxPool: torch.Size([64, 16, 15, 15])
  After Conv2  : torch.Size([64, 32, 13, 13])
  After Conv3  : torch.Size([64, 64, 11, 11])
  After Conv1  : torch.Size([64, 16, 30, 30])
  After MaxPool: torch.Size([64, 16, 15, 15])
  After Conv2  : torch.Size([64, 32, 13, 13])
  After Conv3  : torch.Size([64, 64, 11, 11])
  After Conv1  : torch.Size([64, 16, 30, 30])
  After MaxPool: torch.Size([64, 16, 15, 15])
  After Conv2  : torch.Size([64, 32, 13, 13])
  After Conv3  : torch.Size([64, 64, 11, 11])
  After Conv1  : torch.Size([64, 16, 30, 30]